# Exact Sobol32 on TPU — ENINCA prototype

128 dimensions, 32-bit words, exported SciPy parameters. CPU and interpreted Pallas validation completed; native TPU compilation and performance remain unverified. No TPU or application speedup claimed. The JAX comparator also includes block optimization. Select **Runtime > Change runtime type > TPU**, then execute the four cells. This notebook uses its runner’s account resources. It does not reserve a TPU or automatically transmit results.

## 1. Check TPU access
Installs pinned JAX, NumPy and SciPy if needed. Displays and saves complete error messages. On a Colab quota or missing TPU, do not force CPU fallback: download diagnostics in step 4 and resume when access is restored.

In [ ]:
import datetime, importlib.metadata, os, subprocess, sys
from pathlib import Path
TPU_READY = False
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
PREFLIGHT_LOG = Path.cwd() / ("sobol-tpu-preflight-" + stamp + ".log")
PREFLIGHT_LOG.write_text("Python: " + sys.version + "\n")

def logged(command, env=None):
    with PREFLIGHT_LOG.open("a") as log:
        with subprocess.Popen(command, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True) as process:
            try:
                for line in process.stdout:
                    print(line, end="", flush=True)
                    log.write(line)
                    log.flush()
                code = process.wait()
            except BaseException:
                process.terminate()
                process.wait()
                raise
        log.write("Exit: " + str(code) + "\n")
    if code:
        raise RuntimeError("TPU setup failed. Full log above and in " + str(PREFLIGHT_LOG)
                           + ". Use cell 4 to download diagnostics. No benchmark started.")

required = {"jax": "0.10.2", "jaxlib": "0.10.2", "numpy": "2.4.6", "scipy": "1.17.1"}
try:
    ready = all(importlib.metadata.version(k) == v for k, v in required.items())
    ready = ready and bool(importlib.metadata.version("libtpu"))
except importlib.metadata.PackageNotFoundError:
    ready = False
if not ready:
    logged([sys.executable, "-m", "pip", "install", "--upgrade", "jax[tpu]==0.10.2",
            "numpy==2.4.6", "scipy==1.17.1"])
probe = "import jax; assert jax.default_backend() == 'tpu'; print(jax.__version__, jax.devices())"
logged([sys.executable, "-u", "-c", probe], env={**os.environ, "JAX_PLATFORMS": "tpu"})
TPU_READY = True


## 2. Extract sources
The kit is embedded and its hash is checked. No GitHub access is needed. It includes French and English documentation.

In [ ]:
import base64, hashlib, io, json, tarfile, datetime
from pathlib import Path
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '25dacbec0888c1e9771bba6c9209afebfb055820acdbbacaff4174159dc3654a'
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
KIT = Path("/content") / ("eninca-sobol-" + stamp)
KIT.mkdir(exist_ok=False)
with tarfile.open(fileobj=io.BytesIO(payload), mode="r:gz") as tar:
    for entry in tar.getmembers():
        target = (KIT / entry.name).resolve()
        if not entry.isfile() or not target.is_relative_to(KIT.resolve()):
            raise ValueError("Unexpected archive entry: " + entry.name)
        if target.exists():
            raise ValueError("Duplicate archive entry: " + entry.name)
        target.parent.mkdir(parents=True, exist_ok=True)
        with tar.extractfile(entry) as source:
            target.write_bytes(source.read())
manifest = json.loads((KIT / "manifest.json").read_text())
for rel, expected in manifest.items():
    assert hashlib.sha256((KIT / rel).read_bytes()).hexdigest() == expected, rel
print("Source hashes PASS:", len(manifest))


## 3. Verify and measure
The `QUICK=True` pilot uses two volumes. Conformance and all compilations precede timing and may take several minutes. After a successful pilot, `QUICK=False` runs the fuller campaign with five new confirmation cases. Every timed output is checked exactly against its reference. Setup and initial uploads are recorded separately from reused-parameter throughput.

In [ ]:
import subprocess, sys, datetime
if not globals().get("TPU_READY", False):
    raise RuntimeError("Cell 1 must complete successfully before the TPU run.")
# True: first native pilot. False: 3 volumes, 3 search + 5 confirmation seeds.
QUICK = True
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RESULTS = KIT / "results" / (stamp + "-sobol-tpu")
command = [sys.executable, "-u", str(KIT / "applications/sobol/run.py"),
           "--backend", "tpu", "--out", str(RESULTS)]
if QUICK:
    command.append("--quick")
print("Conformance, compilation, search, then independent confirmation.", flush=True)
with (KIT / "run.log").open("w") as log:
    with subprocess.Popen(command, cwd=KIT, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            process.terminate()
            process.wait()
            raise
print("Results:", RESULTS, flush=True)
if code:
    raise RuntimeError("Native trial failed. Use cell 4 to download diagnostics.")


## 4. Download results or diagnostics
Also run after failure of the first or third cell. Results include parameters, raw times, configuration selections, versions and hashes. CPU or interpreter output is never a native TPU measurement.

In [ ]:
import tarfile
from google.colab import files
preflight = globals().get("PREFLIGHT_LOG")
kit = globals().get("KIT")
results = globals().get("RESULTS")
if not globals().get("TPU_READY", False) and preflight is not None and preflight.exists():
    files.download(str(preflight))
elif results is not None and results.exists():
    archive = results.with_name(results.name + ".tar.gz")
    with tarfile.open(archive, "w:gz") as tar:
        tar.add(results, arcname=results.name)
        for name in ("run.log", "manifest.json"):
            if kit is not None and (kit / name).exists():
                tar.add(kit / name, arcname=name)
        if preflight is not None and preflight.exists():
            tar.add(preflight, arcname=preflight.name)
    files.download(str(archive))
elif preflight is not None and preflight.exists():
    files.download(str(preflight))
else:
    raise RuntimeError("No diagnostics found. Run cell 1 first.")
